<center>
<a href="https://nxv.fr"> <img src="https://nxv.fr/img/logo-NXV-fondtransparent.png" width="400"> </a>
</center>

# 🦾 Start your project with the SO101 arm 
Thanks to this notebook, you will run your first SO101 Arm environment with the Gym module, and visualize a first dummy operator. 

First, as suggested in [the original SO101 Nexus repo](https://github.com/johnsutor/so101-nexus), we install modules and libraries for the environment and its visualization. 

In [ ]:
# Remove " > /dev/null 2>&1" to see what is going on under the hood
!pip install pyvirtualdisplay > /dev/null 2>&1
!apt-get install -y xvfb python-opengl ffmpeg > /dev/null 2>&1
!pip install so101-nexus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.1/839.1 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 65.2 MB/s eta 0:00:00


In [ ]:
# All imports

import gymnasium as gym
import so101_nexus.mujoco
from gym import logger as gymlogger
from gym.wrappers.record_video import RecordVideo
gymlogger.set_level(40) #error only
import numpy as np
import random
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
import math
import glob
import io
import base64
from IPython.display import HTML
from IPython import display as ipythondisplay


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


These functions will help us visualizing the scene directly into the notebook:

In [ ]:
def show_video():
  mp4list = glob.glob('video/*.mp4')
  if len(mp4list) > 0:
    mp4 = mp4list[0]
    video = io.open(mp4, 'r+b').read()
    encoded = base64.b64encode(video)
    ipythondisplay.display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
  else:
    print("Could not find video")


def wrap_env(env):
  env = RecordVideo(env, './video')
  return env

Now let's start the display:

In [3]:
from pyvirtualdisplay import Display
display = Display(visible=0, size=(1400, 900))
display.start()


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


We now create the Gym environment and test visualization:

In [ ]:
# First config the scene as required
from so101_nexus import PickConfig, CubeObject
config = PickConfig(
    objects=[
        CubeObject(color="green"),
        CubeObject(color="blue"),
        CubeObject(color="yellow"),
        CubeObject(color="orange"),
    ],
    n_distractors=3,
    min_object_separation=0.04,
)

# Then we create the env, and display a dummy agent
# Note the use of the wrap_env() function to encapsulate the gym env
env = wrap_env(gym.make("MuJoCoPickAndPlace-v1", render_mode="rgb_array", config=config, max_episode_steps=256))
obs, info = env.reset()

for _ in range(256):
    state, reward, done, info = env.step(env.action_space.sample())
    if done:
        obs, info = env.reset()
    env.render()

env.close()
show_video()

KeyboardInterrupt: 

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import keras
import tensorflow as tf

#env = gym.make("MuJoCoPickAndPlace-v1", render_mode="rgb_array")

obs, info = env.reset()

n_inputs = env.observation_space.shape
n_outputs = env.action_space.shape[0]

model = keras.models.Sequential([
    keras.layers.Input(n_inputs),
    keras.layers.Dense(32, activation="elu"),
    keras.layers.Dense(32, activation="elu"),
    keras.layers.Dense(n_outputs)
])

def epsilon_greedy_policy(state, epsilon=0):
    if np.random.rand() < epsilon:
        return np.random.randint(n_outputs)
    else:
        Q_values = model.predict(state[np.newaxis], verbose=0)
        return np.argmax(Q_values[0])

from collections import deque
replay_memory = deque(maxlen=2000)

def sample_experiences(batch_size):
    indices = np.random.randint(len(replay_memory), size=batch_size)
    batch = [replay_memory[index] for index in indices]
    states, actions, rewards, next_states, dones = [
        np.array([experience[field_index] for experience in batch])
        for field_index in range(5)]
    return states, actions, rewards, next_states, dones

def play_one_step(env, state, epsilon):
    action = epsilon_greedy_policy(state, epsilon)
    next_state, reward, done, truncated, info = env.step(action)
    # On stocke dans la mémoire de l'agent
    replay_memory.append((state, action, reward, next_state, done))
    return next_state, reward, done, info

batch_size = 32
gamma = 0.95
optimizer = keras.optimizers.Adam(learning_rate=6e-3)
loss_fn = keras.losses.Huber()

def training_step(batch_size):
    experiences = sample_experiences(batch_size)
    states, actions, rewards, next_states, dones = experiences
    next_Q_values = model.predict(next_states, verbose=0)
    max_next_Q_values = np.max(next_Q_values, axis=1)
    target_Q_values = (rewards +
                       (1 - dones) * gamma * max_next_Q_values)
    target_Q_values = target_Q_values.reshape(-1, 1)
    mask = tf.one_hot(actions, n_outputs)
    with tf.GradientTape() as tape:
        all_Q_values = model(states)
        Q_values = tf.reduce_sum(all_Q_values * mask, axis=1, keepdims=True)
        loss = tf.reduce_mean(loss_fn(target_Q_values, Q_values))
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))









Dict('agent_pos': Box(-inf, inf, (4, 6), float32), 'environment_state': Box(-inf, inf, (4, 43), float32))


In [24]:
rewards = []
best_score = 0
n_episodes = 50

for episode in range(n_episodes):
    obs = env.reset()
    obs = obs[0]
    for step in range(200):
        epsilon = max(1 - episode / 500, 0.01)
        obs, reward, done, info = play_one_step(env, obs, epsilon)
        if done:
            break
    rewards.append(step)
    if step >= best_score:
        best_weights = model.get_weights()
        best_score = step
    print("\rEpisode: {}, Steps: {}, eps: {:.3f}".format(episode, step + 1, epsilon), end="")
    if episode > 50:
        training_step(batch_size)

model.set_weights(best_weights)
model.save("mymodel.keras")

plt.figure(figsize=(8, 4))
plt.plot(rewards, "r-")
plt.xlabel("Episode", fontsize=14)
plt.ylabel("Reward accumulation", fontsize=14)
plt.show()

TypeError: Unable to iterate over the following elements: 0

## 📚 Exercices

### 1. Configuration
Refering to the [SO101 documentation](https://so101-nexus.com/docs/environments), change the configuration of the environment and test differents scenarios

### 2. Create your reinforcement model
Depending on your skills, make a model learn the correct actions from rewards and observations, all provided by the Gym environment

## Licensing

[![License: CC BY-NC-SA 4.0](https://img.shields.io/badge/License-CC%20BY--NC--SA%204.0-lightgrey.svg)](https://creativecommons.org/licenses/by-nc-sa/4.0/)

Copyright © 2026 [NXV](https://nxv.fr).

This material is licensed under the
[Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License](https://creativecommons.org/licenses/by-nc-sa/4.0/).